# Import

In [20]:
import pandas as pd
from datasets import load_dataset
from datasets import Dataset
from presidio_analyzer import AnalyzerEngine, RecognizerRegistry
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_analyzer.predefined_recognizers import SpacyRecognizer

# Get hugging face dataset

In [2]:
# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("ANR-MALADES/MediQAl", "mcqu")
df = ds["train"].to_pandas()

Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields
Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields
Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


# Analyse dataset

In [3]:
df.head()

,id,clinical_case,question,answer_a,answer_b,answer_c,answer_d,answer_e,correct_answers,task,medical_subject,question_type
0,7543,"Monsieur R. âgé de 37 ans, forestier de son mé...",Au sujet des vaccinations :,Le vaccin contre la fièvre jaune est obligatoire,Le vaccin contre la fièvre jaune peut se faire...,Le vaccin contre le choléra est un vaccin qui ...,Le vaccin contre l'hépatite B est conseillé ma...,Les vaccinations contre le tétanos et la polio...,A,QCU,Infectious Diseases,Understanding
1,10495,NaN,"Devant une tuméfaction cervicale médiane, juxt...",Kyste dermoïde,Laryngocèle,Kyste du tractus thyréoglosse,Grenouillette sublinguale,Lithiase salivaire,C,QCU,Otorhinolaryngology (ENT),Understanding
2,17342,Une jeune femme de 24 ans vous consulte pour a...,Le traitement de cette patiente fait appel à :,Chirurgie en urgence,Iode radio-actif en urgence,Iode radio-actif en préparation à la chirurgie,Administration d'antithyroïdiens,Aucun de ces traitements,D,QCU,Endocrinology and Metabolism,Reasoning
3,22883,"Un nouveau-né de sexe masculin, âgé de 3 jours...",Chez ce nouveau-né le bilan biologique à montr...,Ictère par Incompatibilité Rhésus,Ictère par déficit en G6PD,ictère infectieux,Ictère par incompatibilité ABO,Ictère Infectieux,D,QCU,Pediatrics,Understanding
4,17514,Un homme de 63 ans est hospitalisé pour des do...,Pour confirmer le diagnostic de maladie d'Addi...,Test de stimulation par la 1.24 corticotrophin...,Test de freinage par la dexaméthasone,Test de stimulation de l'activité rénine plasm...,Test de restriction hydrique,Scintigraphie surrénale à l'iodocholestérol,A,QCU,Endocrinology and Metabolism,Reasoning


In [4]:
# Print distinct values of "medical_subject" column
distinct_medical_subjects = df["medical_subject"].unique()
print(f"Distinct values of 'medical_subject' column: {distinct_medical_subjects}")

Distinct values of 'medical_subject' column: <ArrowStringArray>
[             'Infectious Diseases',        'Otorhinolaryngology (ENT)',
     'Endocrinology and Metabolism',                       'Pediatrics',
                     'Microbiology',                      'Orthopedics',
                     'Biochemistry',                       'Cardiology',
        'Gynecology and Obstetrics',                        'Radiology',
                      'Dermatology',                      'Pulmonology',
                       'Psychiatry',                     'Rheumatology',
          'Hepato-Gastroenterology',                       'Hematology',
                        'Neurology',                       'Immunology',
                    'Ophthalmology',                     'Parasitology',
                   'Nephro-Urology',             'Pathological Anatomy',
            'Occupational Medicine', 'Forensic Medicine and Toxicology',
                   'Rehabilitation',                       '

In [5]:
# Keep only the rows where "question_type" is "Reasoning"
# reasoning_df = df[df["question_type"] == "Reasoning"]
# print(f"Number of rows in reasoning_df: {reasoning_df.shape[0]}")

# Keep only the rows where "clinical_case" and "question" are not null in df
df = df.dropna(subset=["clinical_case", "question"])
print(f"Number of rows in reasoning_df after dropping null values: {df.shape[0]}")

Number of rows in reasoning_df after dropping null values: 3935


In [6]:
df.head()

,id,clinical_case,question,answer_a,answer_b,answer_c,answer_d,answer_e,correct_answers,task,medical_subject,question_type
0,7543,"Monsieur R. âgé de 37 ans, forestier de son mé...",Au sujet des vaccinations :,Le vaccin contre la fièvre jaune est obligatoire,Le vaccin contre la fièvre jaune peut se faire...,Le vaccin contre le choléra est un vaccin qui ...,Le vaccin contre l'hépatite B est conseillé ma...,Les vaccinations contre le tétanos et la polio...,A,QCU,Infectious Diseases,Understanding
2,17342,Une jeune femme de 24 ans vous consulte pour a...,Le traitement de cette patiente fait appel à :,Chirurgie en urgence,Iode radio-actif en urgence,Iode radio-actif en préparation à la chirurgie,Administration d'antithyroïdiens,Aucun de ces traitements,D,QCU,Endocrinology and Metabolism,Reasoning
3,22883,"Un nouveau-né de sexe masculin, âgé de 3 jours...",Chez ce nouveau-né le bilan biologique à montr...,Ictère par Incompatibilité Rhésus,Ictère par déficit en G6PD,ictère infectieux,Ictère par incompatibilité ABO,Ictère Infectieux,D,QCU,Pediatrics,Understanding
4,17514,Un homme de 63 ans est hospitalisé pour des do...,Pour confirmer le diagnostic de maladie d'Addi...,Test de stimulation par la 1.24 corticotrophin...,Test de freinage par la dexaméthasone,Test de stimulation de l'activité rénine plasm...,Test de restriction hydrique,Scintigraphie surrénale à l'iodocholestérol,A,QCU,Endocrinology and Metabolism,Reasoning
5,17623,Une femme de 61 ans vient consulter pour asthé...,Parmi les propositions suivantes concernant le...,L'effondrement de la fixation thyroïdienne suf...,Un taux plasmatique de T3 abaissé suffit à aff...,Il n'y a pas d'hypothyroïdie primitive sans él...,La prolactine peut être élevée,La FSH est basse et s'élève sous métopirone,D,QCU,Endocrinology and Metabolism,Understanding


# Build SFT JSONL file from dataset

In [7]:

SYSTEM_PROMPT = "Tu es un assistant médical expert. Réponds de façon précise et concise."


def to_messages(row) -> list[dict]:
    answers = [row[f"answer_{l.strip().lower()}"] for l in row["correct_answers"].split(",")]
    answer = answers[0] if len(answers) == 1 else "\n".join(f"- {a}" for a in answers)
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{row['clinical_case']}\n\n{row['question']}"},
        {"role": "assistant", "content": answer},
    ]


df["messages"] = df.apply(to_messages, axis=1)

sft_ds = Dataset.from_pandas(
    df[["id", "medical_subject", "question_type", "messages"]], preserve_index=False
)
sft_ds.to_json("sft_mediqal_mcqu.jsonl", force_ascii=False)
sft_ds[0]["messages"]


Creating json from Arrow format: 100%|██████████| 4/4 [00:00<00:00, 64.37ba/s]


[{'role': 'system',
  'content': 'Tu es un assistant médical expert. Réponds de façon précise et concise.'},
 {'role': 'user',
  'content': "Monsieur R. âgé de 37 ans, forestier de son métier, doit partir travailler pendant 2 ans au Gabon dans la région de Lambaréné. Monsieur R. a l'intention de se faire accompagner de son épouse âgée de 36 ans et de son fils âgé de 5 ans. Ils logeront à quelques kilomètres de la forêt mais Mr R., de par son métier, passera la plupart de son temps au milieu des bois. Ils consultent 4 mois avant le départ pour se renseigner au sujet des vaccinations et de la chimioprophylaxie du paludisme. Vous lui expliquez les conseils d'hygiène, les éléments de prévention essentiels et les différents modes de contamination des maladies présentes dans la région\n\nAu sujet des vaccinations :"},
 {'role': 'assistant',
  'content': 'Le vaccin contre la fièvre jaune est obligatoire'}]

In [8]:
print(sft_ds)

Dataset({
    features: ['id', 'medical_subject', 'question_type', 'messages'],
    num_rows: 3935
})


# Anonymisation des données

In [9]:
from presidio_analyzer import AnalyzerEngine, PatternRecognizer


In [28]:
# Define text_to_anonymize with the 200 first rows of sft_ds (user message: clinical case + question)
text_to_anonymize = [messages[1]["content"] for messages in sft_ds[:200]["messages"]]

In [29]:
print(f"{len(text_to_anonymize)} textes")
print(text_to_anonymize[0])

200 textes
Monsieur R. âgé de 37 ans, forestier de son métier, doit partir travailler pendant 2 ans au Gabon dans la région de Lambaréné. Monsieur R. a l'intention de se faire accompagner de son épouse âgée de 36 ans et de son fils âgé de 5 ans. Ils logeront à quelques kilomètres de la forêt mais Mr R., de par son métier, passera la plupart de son temps au milieu des bois. Ils consultent 4 mois avant le départ pour se renseigner au sujet des vaccinations et de la chimioprophylaxie du paludisme. Vous lui expliquez les conseils d'hygiène, les éléments de prévention essentiels et les différents modes de contamination des maladies présentes dans la région

Au sujet des vaccinations :


## Vérification de la configuration des entitées


In [ ]:
from presidio_analyzer import AnalyzerEngine
from presidio_analyzer.nlp_engine import NlpEngineProvider

# 1. Définir la configuration pour charger l'anglais et le français
configuration = {
    "nlp_engine_name": "spacy",
    "models": [
        {"lang_code": "en", "model_name": "en_core_web_sm"},
        {"lang_code": "fr", "model_name": "fr_core_news_md"}
    ],
}

# 2. Créer le fournisseur de moteur NLP avec cette configuration
provider = NlpEngineProvider(nlp_configuration=configuration)
nlp_engine = provider.create_engine()

# 3. Passer ce moteur à l'AnalyzerEngine
analyzer = AnalyzerEngine(nlp_engine=nlp_engine)

# 4. Lister les entités supportées pour le français
entites_fr = analyzer.get_supported_entities(language="fr")
print("Entités supportées en français :")
print(entites_fr)


Entités supportées en français :
['ID', 'NRP', 'PHONE_NUMBER', 'EMAIL', 'ORGANIZATION', 'LOCATION', 'AGE', 'PERSON', 'DATE_TIME']


In [ ]:
# 1. Moteur NLP avec le modèle spaCy français
nlp_engine = NlpEngineProvider(nlp_configuration={
    "nlp_engine_name": "spacy",
    "models": [{"lang_code": "fr", "model_name": "fr_core_news_md"}],
}).create_engine()

# 2. Registre de reconnaisseurs déclaré pour le français
registry = RecognizerRegistry(supported_languages=["fr"])
registry.add_recognizer(SpacyRecognizer(supported_language="fr", supported_entities=["PERSON"]))

# 3. Analyseur configuré pour le français
analyzer = AnalyzerEngine(nlp_engine=nlp_engine, registry=registry, supported_languages=["fr"])

analyzer_results = analyzer.analyze(text=text_to_anonymize, entities=["PERSON"], language="fr")
print(f"Analyzer results for text: {text_to_anonymize}")
print(analyzer_results)

# Parse the analyzer results to extract the detected entities
detected_entities = []
for result in analyzer_results:
    entity = {
        "entity_type": result.entity_type,
        "start": result.start,
        "end": result.end,
        "score": result.score,
        "text": text_to_anonymize[result.start:result.end],
    }
    detected_entities.append(entity)

print(f"Detected entities: {detected_entities}")


Entités supportées en français :
['PERSON']
Analyzer results for text: Monsieur D. Daniel, âgé de 38 ans, a vu apparaître depuis quelques semaines une dyspnée d'effort progressive et une augmentation du volume de l'abdomen. L'interrogatoire apprend que le patient a présenté des crises de RAA à l'âge de 14 et 16 ans, traitées par pénicilline et corticoïdes, une primo-infection tuberculeuse sévère à l'âge de 20 ans, et enfin un accident de la circulation avec fractures pluri-costales antérieures et hémothorax gauche à l'âge de 25 ans. L'auscultation cardiaque révèle une arythmie complète récente et un 3ème bruit proto-diastolique. La TA est à 11/8. Les radiographies thoraciques montrent un coeur de volume normal, un épanchement pleural droit avec séquelles de la base gauche, des hiles épaissis, des nodules calcifiés du lobe supérieur gauche. Il existe enfin des calcifications péricardiques nettement visibles sur les incidences transverse et oblique. L'ECG confirme la fibrillation auricul

In [31]:
# Define text_to_anonymize with the 200 first rows of sft_ds (user message: clinical case + question)
text_to_anonymize = [messages[1]["content"] for messages in sft_ds[:200]["messages"]]
print(text_to_anonymize)

["Monsieur R. âgé de 37 ans, forestier de son métier, doit partir travailler pendant 2 ans au Gabon dans la région de Lambaréné. Monsieur R. a l'intention de se faire accompagner de son épouse âgée de 36 ans et de son fils âgé de 5 ans. Ils logeront à quelques kilomètres de la forêt mais Mr R., de par son métier, passera la plupart de son temps au milieu des bois. Ils consultent 4 mois avant le départ pour se renseigner au sujet des vaccinations et de la chimioprophylaxie du paludisme. Vous lui expliquez les conseils d'hygiène, les éléments de prévention essentiels et les différents modes de contamination des maladies présentes dans la région\n\nAu sujet des vaccinations :", "Une jeune femme de 24 ans vous consulte pour asthénie, diarrhée, palpitations, sueurs. A l'examen, le pouls de repos est à 110/mn, le corps thyroïde est globalement hypertrophié, la tension artérielle est de 180-80 mm Hg. Les réflexes achilléens sont vifs. On note également une prise de poids de 3 kilos en un mois

In [30]:
# Analyse des 200 textes avec l'analyseur français défini dans la cellule précédente
rows = []
for index, text in enumerate(text_to_anonymize):
    for result in analyzer.analyze(text=text, entities=["PERSON"], language="fr"):
        rows.append(
            {
                "row": index,
                "entity_type": result.entity_type,
                "score": round(result.score, 2),
                "start": result.start,
                "end": result.end,
                "text": text[result.start : result.end],
            }
        )

detections = pd.DataFrame(rows)
print(f"{detections['row'].nunique()} textes sur {len(text_to_anonymize)} contiennent au moins une entité PERSON")
print(f"{len(detections)} entités PERSON détectées")
detections

74 textes sur 200 contiennent au moins une entité PERSON
134 entités PERSON détectées


,row,entity_type,score,start,end,text
0,0,PERSON,0.85,127,138,Monsieur R.
1,0,PERSON,0.85,287,292,Mr R.
2,2,PERSON,0.85,413,424,Fibrinogène
3,2,PERSON,0.85,456,463,retenez
4,3,PERSON,0.85,1710,1714,Céto
...,...,...,...,...,...,...
129,193,PERSON,0.85,585,590,Denis
130,193,PERSON,0.85,1498,1503,Denis
131,194,PERSON,0.85,819,825,mitra!
132,195,PERSON,0.85,793,795,Ht
